<a href="https://colab.research.google.com/github/kiriakosgp/papadopoulos_av_analysis/blob/main/late_fusion_image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
import json

with open("/content/drive/MyDrive/multimodal_baselines.json", "r") as f:
    data = json.load(f)

# Keep only what we need
records = []
for item in data:
    records.append({
        "video_id": item["video_id"],
        "label": item["thumbnail_pred"],
        "confidence": item["thumbnail_score"]
    })

In [ ]:
LABEL_MAP = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

for r in records:
  if r["label"] is not None:
    r["label"] = LABEL_MAP[r["label"]]

In [ ]:
CONF_THRESH = 0.63

filtered = [
    r for r in records
    if r["confidence"] is not None and r["confidence"] >= CONF_THRESH
]

print(f"Kept {len(filtered)} high-confidence samples")

Kept 556 high-confidence samples


In [ ]:
import os

EMB_DIR = "/content/drive/MyDrive/image_embeddings"

X = []
y = []

missing = 0

for r in filtered:
    vid = r["video_id"]
    emb_path = os.path.join(EMB_DIR, f"{vid}.npy")

    if not os.path.exists(emb_path):
        missing += 1
        continue

    emb = np.load(emb_path)

    X.append(emb)
    y.append(r["label"])

X = np.stack(X)
y = np.array(y)

print("Final dataset shape:", X.shape)
print("Missing embeddings:", missing)

Final dataset shape: (556, 512)
Missing embeddings: 0


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_ds = EmbeddingDataset(X_train, y_train)
val_ds   = EmbeddingDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64)

In [ ]:
class ImageSentimentMLP(nn.Module):
    def __init__(self, input_dim=512, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ImageSentimentMLP(
    input_dim=X.shape[1],
    num_classes=len(set(y))
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
def train_epoch():
    model.train()
    total_loss = 0

    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
def evaluate():
    model.eval()
    preds, gold = [], []

    with torch.no_grad():
        for Xb, yb in val_loader:
            logits = model(Xb.to(device))
            pred = torch.argmax(logits, dim=1).cpu().numpy()

            preds.extend(pred)
            gold.extend(yb.numpy())

    return {
        "acc": accuracy_score(gold, preds),
        "f1": f1_score(gold, preds, average="weighted")
    }

In [ ]:
BEST_F1 = 0.0

for epoch in range(10):
    loss = train_epoch()
    metrics = evaluate()

    if metrics["f1"] > BEST_F1:
        BEST_F1 = metrics["f1"]
        torch.save(model.state_dict(), "image_stage1_best.pt")

    print(
        f"Epoch {epoch+1} | "
        f"Loss {loss:.4f} | "
        f"Acc {metrics['acc']:.3f} | "
        f"F1 {metrics['f1']:.3f}"
    )

Epoch 1 | Loss 0.9885 | Acc 0.768 | F1 0.667
Epoch 2 | Loss 0.7040 | Acc 0.768 | F1 0.667
Epoch 3 | Loss 0.5687 | Acc 0.768 | F1 0.667
Epoch 4 | Loss 0.4920 | Acc 0.804 | F1 0.733
Epoch 5 | Loss 0.4136 | Acc 0.839 | F1 0.805
Epoch 6 | Loss 0.3503 | Acc 0.875 | F1 0.857
Epoch 7 | Loss 0.2919 | Acc 0.875 | F1 0.857
Epoch 8 | Loss 0.2506 | Acc 0.875 | F1 0.862
Epoch 9 | Loss 0.2040 | Acc 0.902 | F1 0.894
Epoch 10 | Loss 0.1791 | Acc 0.920 | F1 0.914


In [ ]:
model = ImageSentimentMLP(input_dim=512, num_classes=3).to(device)
model.load_state_dict(torch.load("image_stage1_best.pt"))
model.eval()

ImageSentimentMLP(
  (net): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=3, bias=True)
  )
)

In [ ]:
all_X = []
all_vids = []

for fname in os.listdir(EMB_DIR):
    if not fname.endswith(".npy"):
        continue

    vid = fname.replace(".npy", "")
    emb = np.load(os.path.join(EMB_DIR, fname))

    all_X.append(emb)
    all_vids.append(vid)

all_X = torch.tensor(np.stack(all_X), dtype=torch.float32).to(device)

print("Total samples:", all_X.shape[0])  # should be 1176

Total samples: 1202


In [ ]:
with torch.no_grad():
    logits = model(all_X)
    probs = torch.softmax(logits, dim=1)

conf, preds = probs.max(dim=1)

conf = conf.cpu().numpy()
preds = preds.cpu().numpy()

In [ ]:
NEW_CONF_THRESH = 0.6

mask = conf >= NEW_CONF_THRESH

X_stage2 = all_X.cpu().numpy()[mask]
y_stage2 = preds[mask]

print("Stage-2 samples:", len(y_stage2))

Stage-2 samples: 910


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_stage2,
    y_stage2,
    test_size=0.2,
    stratify=y_stage2,
    random_state=42
)

In [ ]:
model = ImageSentimentMLP(
    input_dim=512,
    num_classes=3
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/results"
os.makedirs(SAVE_PATH, exist_ok=True)

csv_path = os.path.join(SAVE_PATH, "image_stage2_predictions.csv")
json_path = os.path.join(SAVE_PATH, "image_stage2_predictions.json")

INV_LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}
pred_labels_str = [INV_LABEL_MAP[p] for p in preds]

import pandas as pd

results_df = pd.DataFrame({
    "video_id": all_vids,
    "predicted_label": preds,
    "predicted_label_str": pred_labels_str,
    "confidence": conf
})

results_df.to_csv(csv_path, index=False)
print("Saved CSV")

import json

results_list = [
    {
        "video_id": vid,
        "predicted_label": int(pred),
        "predicted_label_str": str(pred_str),
        "confidence": float(conf_val)
    }
    for vid, pred, pred_str, conf_val in zip(all_vids, preds, pred_labels_str, conf)
]

with open(json_path, "w") as f:
    json.dump(results_list, f, indent=2)

print("Saved JSON")

Saved CSV
Saved JSON
